# Enterprise B2B Company Search - V6 End-to-End Demo
Welcome to the `scaling-succotash` platform showcase. This notebook covers data ingestion, baseline deterministic search, intelligent semantic search, agentic routing, and the Singleflight Request Collapsing / Redis Cache optimizations.

## 1. Environment Overview & Imports
Let's import our tools and point them at the backend.

In [ ]:
import httpx
import time
import asyncio
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt

API_URL = 'http://127.0.0.1:8000/api/v2/search'
client = httpx.AsyncClient(timeout=30.0)


## 2. Data Exploration
We can inspect the raw data being ingested into OpenSearch using `polars`.

In [ ]:
df = pl.read_csv('data/companies.csv')
print(f"Total companies: {len(df)}")
df.head(5).to_pandas()


## 3. Deterministic Search (Baseline)
Traditional keyword filtering using strict matching.

In [ ]:
payload = {'industry': 'Software', 'size': 10, 'page': 1}
response = await client.post(API_URL, json=payload)
print(response.json())


## 4. Intelligent Semantic Search
Fuzzy query intent extraction and vector hnsw embeddings.

In [ ]:
payload = {'query': 'Cloud networking startups in California'}
response = await client.post(f"{API_URL}/intelligent", json=payload)
print(response.json())


## 5. Agentic Workflow
Queries with external context automatically get routed to `requires_agent: true`.

In [ ]:
payload = {'query': 'Recent acquisitions by Cisco in the AI space'}
response = await client.post(f"{API_URL}/intelligent", json=payload)
print(response.json())


## 6. Performance Optimization & Cache Stampede Prevention
Testing the V5 optimizations sequentially. Notice the `X-Cache-Hit: true` response header!

In [ ]:
query = 'AI models for edge devices optimizing power'
payload = {'query': query}

start = time.time()
r1 = await client.post(f"{API_URL}/intelligent", json=payload)
t1 = time.time() - start
print(f"First Request (Full Pipeline): {t1:.4f}s - Cache Hit: {r1.headers.get('x-cache-hit', 'false')}")

start = time.time()
r2 = await client.post(f"{API_URL}/intelligent", json=payload)
t2 = time.time() - start
print(f"Second Request (Redis Cache): {t2:.4f}s - Cache Hit: {r2.headers.get('x-cache-hit', 'false')}")


In [ ]:
await client.aclose()
